# Mosquito-borne disease project: data pipeline

This notebook downloads every dataset the dashboards need, cleans it, and gives you **one zip with the 11 cleaned files**.

**How to use it**
1. In Colab: **Runtime → Run all**.
2. Wait about 5–10 minutes. Part A builds every file and downloads `cleaned_data.zip` to your computer.
3. Part B (about 45–60 min, on by default) adds HDB building age for dashboard D12 and then downloads `cleaned_data_final.zip`. Part C (off by default) adds country humidity. **Share the last zip that downloads.**
4. Read the **check report** at the end of Part A. Every line should say PASS. If one says FAIL, read its "fix" line.

Colab first asks to connect to Google Drive: click **Connect to Google Drive** and allow it. Everything is saved in your Drive folder **G1T8_mosquito_data** (`raw` = downloads, `clean` = the 11 files to share).

If a download fails, the error tells you which link to open and which file name to save it as. Put that file in `G1T8_mosquito_data/raw` in your Google Drive, then run all again. Files already downloaded are reused, so a re-run is fast.

## Settings
Change only these switches. Leave everything else as it is.

In [ ]:
SAVE_TO_GOOGLE_DRIVE = True         # Saves everything in your Google Drive folder "G1T8_mosquito_data" (survives disconnects).
RUN_PART_B_HDB_GEOCODING = True     # D12 "Median HDB block age". About 45-60 minutes. Cached after the first run.
RUN_PART_C_ERA5_HUMIDITY = False    # Country humidity (D4, D6, D8). Needs a Copernicus token OR an ERA5 file in raw/ (see Part C).
CDS_API_KEY = ""                     # Only for Part C: paste your Copernicus personal access token between the quotes.

## Setup (installs and helper functions)

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
TEST_MODE = os.environ.get("PIPELINE_TEST") == "1"
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "geopandas", "epiweeks", "openpyxl", "cdsapi", "netCDF4"], check=False)

import io, re, json, time, zipfile, datetime as dt
import numpy as np
import pandas as pd
import requests
import geopandas as gpd
from shapely.geometry import shape
from epiweeks import Week

BASE = os.path.abspath(os.environ.get("PIPELINE_BASE", "mosquito"))
if IN_COLAB:
    BASE = "/content/mosquito"
    if SAVE_TO_GOOGLE_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            BASE = "/content/drive/MyDrive/G1T8_mosquito_data"
        except Exception as e:
            print("Google Drive not connected, saving inside Colab only:", e)
RAW, OUT = f"{BASE}/raw", f"{BASE}/clean"
os.makedirs(RAW, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
Y0, Y1 = 2000, 2024
TODAY = dt.date.today().isoformat()
HEADERS = {"User-Agent": "Mozilla/5.0 (IS428 student project data pipeline)"}
STATUS = []   # (step, ok, message)


def note(step, ok, msg=""):
    STATUS.append((step, ok, msg))
    print(("OK   " if ok else "FAIL ") + step + (f": {msg}" if msg else ""))


def http_get(url, params=None, tries=4, timeout=180):
    """Download with retries. In test mode the request is answered from local fixtures."""
    if TEST_MODE:
        import fixtures
        return fixtures.fake_get(url, params)
    last = None
    for i in range(tries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=timeout)
            if r.status_code == 429:
                time.sleep(10 * (i + 1))
                continue
            r.raise_for_status()
            return r
        except Exception as e:
            last = e
            time.sleep(3 * (i + 1))
    raise RuntimeError(f"Could not download {url} ({last})")


def raw_file(name, url, params=None, manual=None):
    """Download once into raw/ and reuse it afterwards."""
    path = f"{RAW}/{name}"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    try:
        content = http_get(url, params).content
    except Exception as e:
        raise RuntimeError(f"{e}\n   FIX: open {manual or url} in your browser, download the file, "
                           f"rename it to '{name}' and upload it to {RAW}, then run all again.")
    with open(path, "wb") as f:
        f.write(content)
    return path


def raw_json(name, url, params=None, manual=None):
    with open(raw_file(name, url, params, manual), encoding="utf-8") as f:
        return json.load(f)


def datagovsg_table(dataset_id, name):
    """Whole table from data.gov.sg, paged through the datastore API; cached as raw/<name>.csv."""
    path = f"{RAW}/{name}.csv"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return pd.read_csv(path, dtype=str)
    rows, offset = [], 0
    try:
        while True:
            r = http_get("https://data.gov.sg/api/action/datastore_search",
                         {"resource_id": dataset_id, "limit": 5000, "offset": offset}).json()
            recs = r["result"]["records"]
            rows += recs
            offset += len(recs)
            if not recs or offset >= int(r["result"].get("total", offset)):
                break
            time.sleep(1.5)
    except Exception as e:
        raise RuntimeError(f"{e}\n   FIX: open https://data.gov.sg/datasets/{dataset_id}/view, click Download, "
                           f"rename the CSV to '{name}.csv' and upload it to {RAW}, then run all again.")
    df = pd.DataFrame(rows).drop(columns=["_id", "_full_count", "rank"], errors="ignore").astype(str)
    df.to_csv(path, index=False)
    return df


def datagovsg_file(dataset_id, name):
    """A file dataset (e.g. GeoJSON) from data.gov.sg via poll-download; cached as raw/<name>."""
    path = f"{RAW}/{name}"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    api = f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}"
    try:
        url = None
        for attempt in range(6):
            j = http_get(f"{api}/poll-download").json()
            url = (j.get("data") or {}).get("url")
            if url:
                break
            http_get(f"{api}/initiate-download")
            time.sleep(5)
        if not url:
            raise RuntimeError("no download link returned")
        with open(path, "wb") as f:
            f.write(http_get(url).content)
    except Exception as e:
        raise RuntimeError(f"{e}\n   FIX: open https://data.gov.sg/datasets/{dataset_id}/view, click Download, "
                           f"rename the file to '{name}' and upload it to {RAW}, then run all again.")
    return path


def num(s):
    """Text to number; '-', 'na', '' become NaN."""
    return pd.to_numeric(pd.Series(s).astype(str).str.replace(",", "").str.strip()
                         .replace({"-": np.nan, "na": np.nan, "NA": np.nan, "": np.nan, "nan": np.nan}), errors="coerce").values


def find_col(df, *keys):
    """First column whose lower-case name contains every key."""
    for c in df.columns:
        low = str(c).lower()
        if all(k in low for k in keys):
            return c
    return None


def read_geojson_props(path):
    """Read a data.gov.sg GeoJSON. Its attributes may sit in an HTML 'Description' field; unpack them."""
    with open(path, encoding="utf-8") as f:
        gj = json.load(f)
    recs = []
    for ft in gj["features"]:
        props = dict(ft.get("properties") or {})
        desc = props.get("Description") or props.get("description")
        if isinstance(desc, str) and "<td" in desc:
            for k, v in re.findall(r"<th>\s*(.*?)\s*</th>\s*<td>\s*(.*?)\s*</td>", desc, flags=re.S):
                props[k.strip()] = re.sub("<.*?>", "", v).strip()
        props["geometry"] = shape(ft["geometry"]) if ft.get("geometry") else None
        recs.append(props)
    g = gpd.GeoDataFrame(recs, geometry="geometry", crs="EPSG:4326")
    g["geometry"] = gpd.GeoSeries(g.geometry, crs="EPSG:4326").force_2d() if hasattr(gpd.GeoSeries, "force_2d") else g.geometry
    return g


def changi_monthly():
    """Singapore's monthly rainfall, temperature and humidity at the Changi climate station (data.gov.sg)."""
    out = None
    for dataset_id, name, src, col in [
            ("d_b16d06b83473fdfcc92ed9d37b66ba58", "changi_rainfall_monthly", "total_rainfall", "precip_mm"),
            ("d_755290a24afe70c8f9e8bcbf9f251573", "changi_temperature_monthly", "mean_temp", "temp_c"),
            ("d_e3b1111a26b2aefef4f386c068d4ca5d", "changi_humidity_monthly", "mean_rh", "rh_pct")]:
        t = datagovsg_table(dataset_id, name)
        f = pd.DataFrame({"month": pd.to_datetime(t["month"], format="%Y-%m"), col: num(t[src])})
        out = f if out is None else out.merge(f, on="month", how="outer")
    return out


def fill_singapore_climate(g):
    """The global climate averages skip Singapore (too small for the grid), so its blank years are filled
    from the Changi station. climate_source records where each country's climate values come from."""
    g = g.copy()
    if "climate_source" not in g.columns:
        g["climate_source"] = np.where(g["temp_c"].notna() | g["precip_mm"].notna(), "ERA5 country average (OWID)", None)
    try:
        cm = changi_monthly()
        cm["year"] = cm["month"].dt.year
        cy = cm.groupby("year").agg(temp_c=("temp_c", "mean"), precip_mm=("precip_mm", "sum"), rh_pct=("rh_pct", "mean"),
                                    n_temp=("temp_c", "count"), n_rain=("precip_mm", "count"), n_rh=("rh_pct", "count"))
        for col, n in [("temp_c", "n_temp"), ("precip_mm", "n_rain"), ("rh_pct", "n_rh")]:
            cy.loc[cy[n] < 12, col] = np.nan          # only whole years
        sg = g["iso3"] == "SGP"
        filled = sg & g["temp_c"].isna() & g["year"].map(cy["temp_c"]).notna()
        for col in ["temp_c", "precip_mm", "rh_pct"]:
            gap = sg & g[col].isna()
            g.loc[gap, col] = g.loc[gap, "year"].map(cy[col])
        g.loc[filled, "climate_source"] = "Changi climate station (data.gov.sg)"
    except Exception as e:
        print("Singapore climate not filled:", e)
    return g


print("Raw files go to:", RAW)
print("Clean files go to:", OUT)

# Part A: build all 11 files (about 5–10 minutes)
## A1. Dengue: OpenDengue V1.3 (reported cases)

In [ ]:
OD_BASE = "https://github.com/OpenDengue/master-repo/raw/main/data/releases/V1.3"
dengue_year = dengue_month = od_names = sg_od_weekly = None
try:
    with zipfile.ZipFile(raw_file("National_extract_V1_3.zip", f"{OD_BASE}/National_extract_V1_3.zip")) as z:
        od = pd.read_csv(z.open([n for n in z.namelist() if n.endswith(".csv")][0]),
                         parse_dates=["calendar_start_date", "calendar_end_date"])
    od["days"] = (od["calendar_end_date"] - od["calendar_start_date"]).dt.days + 1
    od = od[od["Year"].between(Y0, Y1)]
    od_names = od.groupby("ISO_A0")["adm_0_name"].first().str.title()

    dengue_year = (od.groupby(["ISO_A0", "Year"], as_index=False)
                   .agg(dengue_cases=("dengue_total", "sum"), days_covered=("days", "sum"),
                        dengue_case_def=("case_definition_standardised", "first"))
                   .rename(columns={"ISO_A0": "iso3", "Year": "year"}))
    dengue_year["dengue_complete"] = (dengue_year["days_covered"] >= 350).astype(int)
    dengue_year.loc[dengue_year["dengue_complete"] == 0, "dengue_cases"] = np.nan
    dengue_year = dengue_year.drop(columns="days_covered")

    sub = od[od["T_res"].isin(["Week", "Month"])].copy()
    sub["month"] = sub["calendar_start_date"].dt.to_period("M").dt.to_timestamp()
    dengue_month = (sub.groupby(["ISO_A0", "month"], as_index=False)["dengue_total"].sum()
                    .rename(columns={"ISO_A0": "iso3", "dengue_total": "dengue_cases"}))
    note("A1 OpenDengue national", True, f"{dengue_year.iso3.nunique()} countries, "
         f"{int(dengue_year.dengue_complete.sum())} complete country-years")
except Exception as e:
    note("A1 OpenDengue national", False, str(e))

# Singapore weekly dengue from the temporal extract (fills the years after the data.gov.sg bulletin ends)
try:
    with zipfile.ZipFile(raw_file("Temporal_extract_V1_3.zip", f"{OD_BASE}/Temporal_extract_V1_3.zip")) as z:
        name = [n for n in z.namelist() if n.endswith(".csv")][0]
        parts = []
        for chunk in pd.read_csv(z.open(name), chunksize=400_000,
                                 usecols=["ISO_A0", "S_res", "T_res", "Year", "dengue_total", "calendar_start_date"]):
            parts.append(chunk[(chunk["ISO_A0"] == "SGP") & (chunk["S_res"] == "Admin0") & (chunk["T_res"] == "Week")])
    sg_od_weekly = pd.concat(parts)
    sg_od_weekly["week_start"] = pd.to_datetime(sg_od_weekly["calendar_start_date"])
    sg_od_weekly = sg_od_weekly.groupby("week_start", as_index=False)["dengue_total"].sum()
    note("A1 OpenDengue Singapore weekly", True, f"{len(sg_od_weekly)} weeks")
except Exception as e:
    note("A1 OpenDengue Singapore weekly", False, str(e))

## A2. Malaria: WHO Global Health Observatory

In [ ]:
def gho(code):
    url = f"https://ghoapi.azureedge.net/api/{code}"
    data = raw_json(f"who_{code}.json", url, manual=url)
    vals = data["value"]
    nxt = data.get("@odata.nextLink")
    while nxt and not TEST_MODE:            # follow paging if WHO splits the result
        more = http_get(nxt).json()
        vals += more["value"]
        nxt = more.get("@odata.nextLink")
    df = pd.DataFrame(vals)
    df = df[df["SpatialDimType"] == "COUNTRY"]
    return (df.rename(columns={"SpatialDim": "iso3", "TimeDim": "year"})[["iso3", "year", "NumericValue"]]
              .groupby(["iso3", "year"], as_index=False)["NumericValue"].mean())

malaria = None
try:
    cases = gho("MALARIA_EST_CASES").rename(columns={"NumericValue": "malaria_cases_est"})
    inc = gho("MALARIA_EST_INCIDENCE").rename(columns={"NumericValue": "malaria_inc_1000_at_risk"})
    malaria = cases.merge(inc, on=["iso3", "year"], how="outer")
    malaria = malaria[malaria["year"].between(Y0, Y1)]
    note("A2 WHO malaria", True, f"{malaria.iso3.nunique()} endemic countries, years {malaria.year.min()}-{malaria.year.max()}")
except Exception as e:
    note("A2 WHO malaria", False, str(e))

## A3. World Bank indicators and country list

In [ ]:
WB = {"SP.POP.TOTL": "population", "EN.POP.DNST": "pop_density", "SP.URB.TOTL.IN.ZS": "urban_pct",
      "NY.GDP.PCAP.PP.KD": "gdp_pc_ppp", "SH.H2O.BASW.ZS": "water_basic_pct",
      "SH.STA.BASS.ZS": "sanitation_basic_pct", "SH.XPD.CHEX.PC.CD": "health_exp_pc"}
countries = wb = None
try:
    meta = raw_json("wb_countries.json", "https://api.worldbank.org/v2/country",
                    {"format": "json", "per_page": 400})[1]
    countries = pd.DataFrame([{"iso3": m["id"], "country": m["name"].strip(),
                               "region": m["region"]["value"].strip(),
                               "income_group": m["incomeLevel"]["value"].strip()}
                              for m in meta if m["region"]["value"].strip() != "Aggregates"])
    frames = []
    for code, col in WB.items():
        recs = raw_json(f"wb_{code}.json", f"https://api.worldbank.org/v2/country/all/indicator/{code}",
                        {"format": "json", "per_page": 20000, "date": f"{Y0}:{Y1}"})[1] or []
        f = pd.DataFrame([{"iso3": r["countryiso3code"], "year": int(r["date"]), col: r["value"]} for r in recs])
        frames.append(f[f["iso3"].isin(countries["iso3"])].set_index(["iso3", "year"]))
    wb = pd.concat(frames, axis=1).reset_index()
    note("A3 World Bank", True, f"{len(countries)} countries, {len(WB)} indicators")
except Exception as e:
    note("A3 World Bank", False, str(e))

## A4. Climate: OWID temperature (monthly) and rainfall (annual), both ERA5

In [ ]:
temp_month = temp_year = precip = None
try:
    t = pd.read_csv(raw_file("owid_monthly_temperature.csv",
                             "https://ourworldindata.org/grapher/average-monthly-surface-temperature.csv",
                             {"v": 1, "csvType": "full", "useColumnShortNames": "true"},
                             manual="https://ourworldindata.org/grapher/average-monthly-surface-temperature (Download → Full data CSV)"))
    vcol = [c for c in t.columns if c not in ("entity", "code", "month", "year", "Entity", "Code", "Day", "Year")][-1]
    tcol = "month" if "month" in t.columns else ("Day" if "Day" in t.columns else find_col(t, "month"))
    t = t[t["code"].astype(str).str.fullmatch(r"[A-Z]{3}")] if "code" in t.columns else t
    t["month"] = pd.to_datetime(t[tcol].astype(str).str[:7], format="%Y-%m").dt.to_period("M").dt.to_timestamp()
    t = t[t["month"].dt.year.between(Y0, Y1)].rename(columns={"code": "iso3", vcol: "temp_c"})
    temp_month = t[["iso3", "month", "temp_c"]]
    ty = temp_month.assign(year=temp_month["month"].dt.year).groupby(["iso3", "year"])["temp_c"].agg(["mean", "count"])
    temp_year = ty[ty["count"] == 12]["mean"].rename("temp_c").reset_index()

    p = pd.read_csv(raw_file("owid_annual_precipitation.csv",
                             "https://ourworldindata.org/grapher/average-precipitation-per-year.csv",
                             {"v": 1, "csvType": "full", "useColumnShortNames": "true"},
                             manual="https://ourworldindata.org/grapher/average-precipitation-per-year (Download → Full data CSV)"))
    pcol = [c for c in p.columns if c not in ("entity", "code", "year", "Entity", "Code", "Year")][-1]
    p = p[p["code"].astype(str).str.fullmatch(r"[A-Z]{3}")]
    precip = p.rename(columns={"code": "iso3", pcol: "precip_mm"})[["iso3", "year", "precip_mm"]]
    precip = precip[precip["year"].between(Y0, Y1)]
    note("A4 OWID climate", True, f"temperature for {temp_year.iso3.nunique()} countries, rainfall for {precip.iso3.nunique()}")
except Exception as e:
    note("A4 OWID climate", False, str(e))

## A5. HDI (UNDP) and poverty (World Bank PIP, $3.00/day)

In [ ]:
hdi = poverty = None
try:
    path = raw_file("HDR25_Composite_indices_complete_time_series.csv",
                    "https://hdr.undp.org/sites/default/files/2025_HDR/HDR25_Composite_indices_complete_time_series.csv")
    try:
        h = pd.read_csv(path)
    except UnicodeDecodeError:
        h = pd.read_csv(path, encoding="latin-1")
    h = h[h["iso3"].astype(str).str.fullmatch(r"[A-Z]{3}")]
    hcols = [c for c in h.columns if re.fullmatch(r"hdi_\d{4}", c)]
    hdi = h.melt(id_vars="iso3", value_vars=hcols, var_name="year", value_name="hdi")
    hdi["year"] = hdi["year"].str[4:].astype(int)
    hdi = hdi[hdi["year"].between(Y0, Y1)]
    note("A5 UNDP HDI", True, f"{hdi.iso3.nunique()} countries")
except Exception as e:
    note("A5 UNDP HDI", False, str(e))

try:
    pv = pd.read_csv(raw_file("pip_poverty_3usd.csv", "https://api.worldbank.org/pip/v1/pip",
                              {"country": "all", "year": "all", "povline": 3.0, "fill_gaps": "false", "format": "csv"}))
    pv = pv[pv["reporting_level"] == "national"]
    poverty = (pv.groupby(["country_code", "reporting_year"], as_index=False)["headcount"].mean()
                 .rename(columns={"country_code": "iso3", "reporting_year": "year"}))
    poverty["poverty_pct"] = poverty.pop("headcount") * 100
    poverty = poverty[poverty["year"].between(Y0, Y1)]
    note("A5 PIP poverty", True, f"{len(poverty)} country-years with a survey")
except Exception as e:
    note("A5 PIP poverty (optional)", False, str(e))

## A6. Humidity (only if Part C was run before; otherwise left blank)

In [ ]:
humidity = None
if os.path.exists(f"{RAW}/era5_rh_country_year.csv"):
    humidity = pd.read_csv(f"{RAW}/era5_rh_country_year.csv")
    note("A6 humidity", True, f"{humidity.iso3.nunique()} countries")
else:
    print("Humidity skipped for now: rh_pct stays blank until Part C has been run once.")

## A7. Merge into global_country_year.csv and global_country_month.csv

In [ ]:
try:
    iso = set(countries["iso3"]) if countries is not None else set()
    for df in (dengue_year, malaria):
        if df is not None:
            iso |= set(df["iso3"])
    grid = pd.MultiIndex.from_product([sorted(iso), range(Y0, Y1 + 1)], names=["iso3", "year"]).to_frame(index=False)
    g = grid
    for df in (countries, dengue_year, malaria, wb, temp_year, precip, humidity, hdi, poverty):
        if df is None:
            continue
        g = g.merge(df, on=["iso3"] if "year" not in df.columns else ["iso3", "year"], how="left")

    # names for territories the World Bank does not list
    if "country" not in g.columns:
        g["country"] = np.nan
    if od_names is not None:
        g["country"] = g["country"].fillna(g["iso3"].map(od_names))
    g["country"] = g["country"].fillna(g["iso3"])

    for col in ["dengue_cases", "dengue_complete", "dengue_case_def", "malaria_cases_est", "malaria_inc_1000_at_risk",
                "population", "pop_density", "urban_pct", "gdp_pc_ppp", "water_basic_pct", "sanitation_basic_pct",
                "health_exp_pc", "temp_c", "precip_mm", "rh_pct", "hdi", "poverty_pct", "region", "income_group"]:
        if col not in g.columns:
            g[col] = np.nan
    g["dengue_complete"] = g["dengue_complete"].fillna(0).astype(int)
    endemic = set(malaria["iso3"]) if malaria is not None else set()
    g["malaria_status"] = np.where(g["iso3"].isin(endemic), "Endemic", "Not endemic")
    g.loc[g["malaria_status"] == "Not endemic", "malaria_cases_est"] = 0
    g["dengue_per_100k"] = g["dengue_cases"] / g["population"] * 1e5
    g["malaria_per_100k"] = g["malaria_cases_est"] / g["population"] * 1e5
    g = fill_singapore_climate(g)

    cols = ["iso3", "country", "region", "income_group", "year", "population",
            "dengue_cases", "dengue_complete", "dengue_case_def", "dengue_per_100k",
            "malaria_status", "malaria_cases_est", "malaria_inc_1000_at_risk", "malaria_per_100k",
            "temp_c", "precip_mm", "rh_pct", "climate_source", "pop_density", "urban_pct", "gdp_pc_ppp", "hdi", "poverty_pct",
            "water_basic_pct", "sanitation_basic_pct", "health_exp_pc"]
    global_year = g[cols].sort_values(["iso3", "year"])
    global_year.to_csv(f"{OUT}/global_country_year.csv", index=False)
    note("A7 global_country_year.csv", True, f"{len(global_year)} rows, {global_year.iso3.nunique()} countries")
except Exception as e:
    global_year = None
    note("A7 global_country_year.csv", False, str(e))

try:
    m = temp_month if temp_month is not None else pd.DataFrame(columns=["iso3", "month", "temp_c"])
    if dengue_month is not None:
        m = m.merge(dengue_month, on=["iso3", "month"], how="outer")
    try:                                   # Singapore's monthly temperature from Changi (the global source skips it)
        cm = changi_monthly()[["month", "temp_c"]].rename(columns={"temp_c": "temp_changi"}).assign(iso3="SGP")
        m = m.merge(cm, on=["iso3", "month"], how="outer")
        m["temp_c"] = m["temp_c"].fillna(m.pop("temp_changi"))
    except Exception as e:
        print("Singapore monthly temperature not filled:", e)
    names =global_year.drop_duplicates("iso3").set_index("iso3")["country"] if global_year is not None else pd.Series(dtype=str)
    m["country"] = m["iso3"].map(names).fillna(m["iso3"])
    m = m[m["month"].dt.year.between(Y0, Y1)]
    m[["iso3", "country", "month", "temp_c", "dengue_cases"]].sort_values(["iso3", "month"]).to_csv(
        f"{OUT}/global_country_month.csv", index=False, date_format="%Y-%m-%d")
    note("A7 global_country_month.csv", True, f"{len(m)} rows")
except Exception as e:
    note("A7 global_country_month.csv", False, str(e))

## A8. Small files: disease_info.csv and gdo_current_season.csv

In [ ]:
pd.DataFrame([
    ["Dengue", "Dengue virus (DENV-1 to 4)", "Aedes aegypti and Aedes albopictus", "https://en.wikipedia.org/wiki/Dengue_fever"],
    ["Malaria", "Plasmodium parasites", "Anopheles mosquitoes", "https://en.wikipedia.org/wiki/Malaria"],
], columns=["disease", "pathogen", "vector", "wiki_url"]).to_csv(f"{OUT}/disease_info.csv", index=False)
note("A8 disease_info.csv", True)

try:
    try:
        gpath = raw_file("global_dengue_data.csv", "https://globaldengueobservatory.org/pages/www/global_dengue_data.csv")
    except Exception:
        gpath = raw_file("global_dengue_data.csv",
                         "https://raw.githubusercontent.com/DengueGlobalObservatory/DENV_global_observatory/main/pages/www/global_dengue_data.csv")
    gdo = pd.read_csv(gpath)
    gdo.to_csv(f"{OUT}/gdo_current_season.csv", index=False)
    note("A8 gdo_current_season.csv", True, f"years {gdo.Year.min()}-{gdo.Year.max()}, {gdo.Country.nunique()} countries")
except Exception as e:
    note("A8 gdo_current_season.csv", False, str(e))

## A9. Singapore weekly cases and population

In [ ]:
sg_weekly = None
try:
    b = datagovsg_table("d_ca168b2cb763640d72c4600a68f9909e", "sg_weekly_bulletin_2012_2022")
    ccol = find_col(b, "case") or "no._of_cases"
    b["epi_year"] = b["epi_week"].str[:4].astype(int)
    b["epi_week_no"] = b["epi_week"].str.extract(r"W(\d+)")[0].astype(int)
    def epi_start(y, w):
        try:
            return Week(int(y), int(w), system="cdc").startdate()
        except ValueError:            # a week 53 the CDC calendar doesn't have: count on from week 1
            return Week(int(y), 1, system="cdc").startdate() + dt.timedelta(days=7 * (int(w) - 1))
    b["week_start"] = [epi_start(y, w) for y, w in zip(b["epi_year"], b["epi_week_no"])]
    b["cases"] = num(b[ccol])
    b["disease"] = b["disease"].str.strip()
    b = b[["epi_year", "epi_week_no", "week_start", "disease", "cases"]].rename(columns={"epi_week_no": "epi_week"})
    parts = [b]
    if sg_od_weekly is not None:
        later = sg_od_weekly[sg_od_weekly["week_start"].dt.year > b["epi_year"].max()].copy()
        wk = [Week.fromdate(d.date(), system="cdc") for d in later["week_start"]]
        later = pd.DataFrame({"epi_year": [w.year for w in wk], "epi_week": [w.week for w in wk],
                              "week_start": later["week_start"].dt.date, "disease": "Dengue (total, OpenDengue)",
                              "cases": later["dengue_total"].values})
        parts.append(later)
    sg_weekly = pd.concat(parts, ignore_index=True)
    sg_weekly["week_start"] = pd.to_datetime(sg_weekly["week_start"])
    sg_weekly.sort_values(["week_start", "disease"]).to_csv(f"{OUT}/sg_weekly_disease.csv", index=False, date_format="%Y-%m-%d")
    note("A9 sg_weekly_disease.csv", True, f"{sg_weekly.epi_year.min()}-{sg_weekly.epi_year.max()}, "
         f"{sg_weekly.disease.nunique()} diseases")
except Exception as e:
    note("A9 sg_weekly_disease.csv", False, str(e))

try:
    pop = wb[wb["iso3"] == "SGP"][["year", "population"]].dropna().rename(columns={"population": "total_population"})
    pop = (pop.set_index("year").reindex(range(int(pop.year.min()), int(pop.year.max()) + 1))
              .interpolate().reset_index().rename(columns={"index": "year"}))
    last = pop.sort_values("year").iloc[-1]
    extra = [{"year": y, "total_population": last["total_population"]} for y in range(int(last["year"]) + 1, 2027)]
    pop = pd.concat([pop, pd.DataFrame(extra)], ignore_index=True)
    pop["note"] = np.where(pop["year"] > last["year"], f"copied from {int(last['year'])}", "World Bank")
    pop["year"] = pop["year"].astype(int)
    pop.to_csv(f"{OUT}/sg_population.csv", index=False)
    note("A9 sg_population.csv", True, f"{pop.year.min()}-{pop.year.max()}")
except Exception as e:
    note("A9 sg_population.csv", False, str(e))

## A10. Singapore monthly weather and the lag table

In [ ]:
try:
    rain = datagovsg_table("d_b16d06b83473fdfcc92ed9d37b66ba58", "changi_rainfall_monthly")
    temp = datagovsg_table("d_755290a24afe70c8f9e8bcbf9f251573", "changi_temperature_monthly")
    rh = datagovsg_table("d_e3b1111a26b2aefef4f386c068d4ca5d", "changi_humidity_monthly")
    weather = []
    for df, col, label in [(rain, "total_rainfall", "Rainfall (mm)"), (temp, "mean_temp", "Temperature (°C)"),
                           (rh, "mean_rh", "Relative humidity (%)")]:
        weather.append(pd.DataFrame({"month": pd.to_datetime(df["month"], format="%Y-%m"),
                                     "weather_var": label, "value": num(df[col])}))
    weather = pd.concat(weather)

    dz = sg_weekly[sg_weekly["disease"].str.contains("Dengue")].copy()
    dz["month"] = (dz["week_start"] + pd.Timedelta(days=3)).dt.to_period("M").dt.to_timestamp()
    dengue_m = dz.groupby("month", as_index=False)["cases"].sum().rename(columns={"cases": "dengue_cases"})

    frames = []
    for lag in range(0, 7):
        w = weather.copy()
        w["month"] = w["month"] + pd.DateOffset(months=lag)      # weather from `lag` months earlier
        f = dengue_m.merge(w, on="month", how="inner").rename(columns={"value": "weather_value_lagged"})
        f["lag_months"] = lag
        frames.append(f)
    lagged = pd.concat(frames)[["month", "lag_months", "weather_var", "weather_value_lagged", "dengue_cases"]]
    lagged.sort_values(["weather_var", "lag_months", "month"]).to_csv(f"{OUT}/sg_monthly_lagged.csv", index=False,
                                                                       date_format="%Y-%m-%d")
    note("A10 sg_monthly_lagged.csv", True, f"{dengue_m.month.min():%Y-%m} to {dengue_m.month.max():%Y-%m}, lags 0-6")
except Exception as e:
    note("A10 sg_monthly_lagged.csv", False, str(e))

## A11. Singapore subzones, Census 2020 profile, 2020 clusters, current NEA clusters

In [ ]:
subz = None
try:
    sp = read_geojson_props(datagovsg_file("d_8594ae9ff96d0c708bc2af633048edfb", "mp2019_subzones_raw.geojson"))
    subz = gpd.GeoDataFrame({"SUBZONE_N": sp["SUBZONE_N"].str.upper().str.strip(),
                             "PLN_AREA_N": sp["PLN_AREA_N"].str.upper().str.strip(),
                             "REGION_N": sp["REGION_N"].str.upper().str.strip()},
                            geometry=sp.geometry, crs="EPSG:4326")
    subz = subz.dissolve(by=["SUBZONE_N", "PLN_AREA_N", "REGION_N"], as_index=False)
    subz["area_km2"] = subz.to_crs(3414).area / 1e6
    if os.path.exists(f"{OUT}/mp2019_subzones.geojson"):
        os.remove(f"{OUT}/mp2019_subzones.geojson")
    subz[["SUBZONE_N", "PLN_AREA_N", "REGION_N", "geometry"]].to_file(f"{OUT}/mp2019_subzones.geojson", driver="GeoJSON")
    note("A11 mp2019_subzones.geojson", True, f"{len(subz)} subzones")
except Exception as e:
    note("A11 mp2019_subzones.geojson", False, str(e))

profile = None
try:
    c = datagovsg_table("d_7f243956483d5901f237e6f87b096636", "census2020_subzone_dwelling")
    c["name"] = c["Number"].str.strip()
    c = c[(c["name"].str.lower() != "total") & ~c["name"].str.contains(" - Total", case=False)]
    tot = num(c["Total"])
    profile = pd.DataFrame({"subzone": c["name"].str.upper().values, "residents_2020": tot,
                            "hdb_pct": num(c["HDBDwellings_Total"]) / tot * 100,
                            "condo_pct": num(c["CondominiumsandOtherApartments"]) / tot * 100,
                            "landed_pct": num(c["LandedProperties"]) / tot * 100})
    profile[["hdb_pct", "condo_pct", "landed_pct"]] = profile[["hdb_pct", "condo_pct", "landed_pct"]].fillna(0)
    profile["residents_2020"] = profile["residents_2020"].fillna(0)
    if subz is not None:
        lookup = subz.drop(columns="geometry").rename(columns={"SUBZONE_N": "subzone", "PLN_AREA_N": "planning_area",
                                                               "REGION_N": "region"})
        unmatched = sorted(set(profile["subzone"]) - set(lookup["subzone"]))
        profile = lookup.merge(profile, on="subzone", how="left")
        profile["residents_2020"] = profile["residents_2020"].fillna(0)
        profile["density_per_km2"] = profile["residents_2020"] / profile["area_km2"]
        if unmatched:
            print("Census names with no matching subzone polygon:", unmatched)
    profile["hdb_median_year_completed"] = np.nan
    profile["hdb_median_age_2020"] = np.nan
    PROFILE_COLS = ["subzone", "planning_area", "region", "residents_2020", "hdb_pct", "condo_pct", "landed_pct",
                    "hdb_median_year_completed", "hdb_median_age_2020", "area_km2", "density_per_km2"]
    for col in PROFILE_COLS:
        if col not in profile.columns:
            profile[col] = np.nan
    profile[PROFILE_COLS].to_csv(f"{OUT}/sg_subzone_profile.csv", index=False)
    note("A11 sg_subzone_profile.csv", True, f"{len(profile)} subzones (building age added by Part B)")
except Exception as e:
    note("A11 sg_subzone_profile.csv", False, str(e))


def to_subzone(points_gdf):
    """Subzone name for each point; points just outside a polygon snap to the nearest one within 200 m."""
    j = gpd.sjoin(points_gdf, subz[["SUBZONE_N", "geometry"]], how="left", predicate="within")
    j = j[~j.index.duplicated()]
    miss = j["SUBZONE_N"].isna()
    if miss.any():
        near = gpd.sjoin_nearest(points_gdf[miss].to_crs(3414), subz[["SUBZONE_N", "geometry"]].to_crs(3414),
                                 how="left", max_distance=200)
        near = near[~near.index.duplicated()]
        j.loc[miss, "SUBZONE_N"] = near["SUBZONE_N"]
    return j["SUBZONE_N"]


def parse_cluster_dates(col):
    """The 2020 cluster file stores each week's end date as a number such as 20200228.
    Read as text and try every common date format; keep the one that puts the most dates in 2020."""
    import warnings
    text = col.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    attempts = [lambda x: pd.to_datetime(x, format="%Y%m%d", errors="coerce"),     # 20200228
                lambda x: pd.to_datetime(x, format="%y%m%d", errors="coerce"),     # 200228
                lambda x: pd.to_datetime(x, errors="coerce"),                      # 2020-02-28, 2/28/2020
                lambda x: pd.to_datetime(x, errors="coerce", dayfirst=True)]       # 28/02/2020
    best, best_share = None, -1.0
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for parse in attempts:
            d = parse(text)
            share = d.between("2020-01-01", "2020-12-31").mean()
            if share > best_share:
                best, best_share = d, share
    return best


try:
    cl = pd.read_csv(raw_file("dengue_outbreak_Singapore_2020.csv", "https://ndownloader.figshare.com/files/36505797",
                              manual="https://doi.org/10.6084/m9.figshare.20417658"))
    lat, lon = find_col(cl, "lat"), find_col(cl, "lon") or find_col(cl, "lng")
    dcol = find_col(cl, "date") or find_col(cl, "week")
    ncol = find_col(cl, "case")
    kcol = find_col(cl, "cluster")
    icol = find_col(cl, "id") if find_col(cl, "id") not in (None, find_col(cl, "subzone")) else None
    when = parse_cluster_dates(cl[dcol])
    clusters = pd.DataFrame({"record_id": cl[icol] if icol else np.arange(1, len(cl) + 1),
                             "lat": num(cl[lat]), "lon": num(cl[lon]), "week_end": when,
                             "case_number": num(cl[ncol]), "cluster_label": cl[kcol] if kcol else np.nan})
    pts = gpd.GeoDataFrame(clusters, geometry=gpd.points_from_xy(clusters["lon"], clusters["lat"]), crs="EPSG:4326")
    clusters["subzone"] = to_subzone(pts).values
    clusters.to_csv(f"{OUT}/sg_clusters_2020.csv", index=False, date_format="%Y-%m-%d")
    note("A11 sg_clusters_2020.csv", True, f"{len(clusters)} records, {clusters.week_end.nunique()} weeks "
         f"({clusters.week_end.min():%d %b} to {clusters.week_end.max():%d %b %Y}), "
         f"{clusters.subzone.notna().mean():.0%} matched to a subzone")
except Exception as e:
    note("A11 sg_clusters_2020.csv", False, str(e))

try:
    nea = read_geojson_props(datagovsg_file("d_dbfabf16158d1b0e1c420627c0819168", f"nea_clusters_{TODAY}.geojson"))
    keep = [c for c in ["LOCALITY", "CASE_SIZE", "HYPERLINK", "NAME"] if c in nea.columns]
    nea = nea[keep + ["geometry"]].copy()
    if "CASE_SIZE" in nea.columns:
        nea["CASE_SIZE"] = num(nea["CASE_SIZE"])
    cent = gpd.GeoDataFrame(geometry=nea.to_crs(3414).centroid.to_crs(4326), crs="EPSG:4326")
    nea["subzone"] = to_subzone(cent).values
    nea["download_date"] = TODAY
    if os.path.exists(f"{OUT}/nea_clusters_now.geojson"):
        os.remove(f"{OUT}/nea_clusters_now.geojson")
    nea.to_file(f"{OUT}/nea_clusters_now.geojson", driver="GeoJSON")
    note("A11 nea_clusters_now.geojson", True, f"{len(nea)} active clusters on {TODAY}")
except Exception as e:
    note("A11 nea_clusters_now.geojson", False, str(e))

## A12. Check report
Every line should say PASS. A FAIL line tells you what to do.

In [ ]:
def check_report():
    rows = []

    def chk(name, cond, fix):
        try:
            ok = bool(cond())
        except Exception:
            ok, fix = False, f"{fix}; the file is missing or empty, so read the FAIL line for that step above"
        rows.append(("PASS" if ok else "FAIL", name, "" if ok else fix))

    gy = pd.read_csv(f"{OUT}/global_country_year.csv") if os.path.exists(f"{OUT}/global_country_year.csv") else None
    chk("global: one row per country-year", lambda: not gy.duplicated(["iso3", "year"]).any(), "re-run A7")
    chk("global: Singapore 2020 dengue = 34,959", lambda: gy.query("iso3=='SGP' and year==2020").dengue_cases.iloc[0] == 34959,
        "OpenDengue file changed or failed (A1)")
    chk("global: 60-110 countries with complete dengue in 2019",
        lambda: 60 <= gy.query("year==2019 and dengue_complete==1").iso3.nunique() <= 110, "check A1")
    chk("global: 70-110 malaria-endemic countries", lambda: 70 <= gy.query("malaria_status=='Endemic'").iso3.nunique() <= 110,
        "check A2 (WHO)")
    chk("global: population for 190+ countries in 2019", lambda: gy.query("year==2019").population.notna().sum() >= 190, "check A3")
    chk("global: temperature for 150+ countries in 2019", lambda: gy.query("year==2019").temp_c.notna().sum() >= 150, "check A4")
    chk("global: HDI for 150+ countries in 2019", lambda: gy.query("year==2019").hdi.notna().sum() >= 150, "check A5")
    chk("global: dengue rates look sane (median 1-1,000 per 100k)",
        lambda: 1 <= gy.dengue_per_100k.median() <= 1000, "population or dengue join is wrong")
    chk("global: Singapore has temperature and rainfall (from Changi)",
        lambda: gy.query("iso3=='SGP' and year==2019")[["temp_c", "precip_mm"]].notna().all(axis=None), "check A7")
    sw = pd.read_csv(f"{OUT}/sg_weekly_disease.csv") if os.path.exists(f"{OUT}/sg_weekly_disease.csv") else None
    chk("Singapore: weekly data from 2012", lambda: sw.epi_year.min() == 2012, "check A9")
    chk("Singapore: 2020 dengue between 30,000 and 40,000",
        lambda: 30000 <= sw[sw.disease.str.contains("Dengue") & (sw.epi_year == 2020)].cases.sum() <= 40000, "check A9")
    lg = pd.read_csv(f"{OUT}/sg_monthly_lagged.csv") if os.path.exists(f"{OUT}/sg_monthly_lagged.csv") else None
    chk("Singapore: lag table has lags 0-6 for 3 weather variables",
        lambda: lg.lag_months.nunique() == 7 and lg.weather_var.nunique() == 3, "check A10")
    pr = pd.read_csv(f"{OUT}/sg_subzone_profile.csv") if os.path.exists(f"{OUT}/sg_subzone_profile.csv") else None
    chk("Singapore: 250+ subzones with residents data", lambda: (pr.residents_2020 > 0).sum() >= 250, "check A11 census")
    cz = pd.read_csv(f"{OUT}/sg_clusters_2020.csv") if os.path.exists(f"{OUT}/sg_clusters_2020.csv") else None
    chk("Singapore: 2020 clusters, 15,000+ records, 90%+ matched to a subzone",
        lambda: len(cz) >= 15000 and cz.subzone.notna().mean() >= 0.9, "check A11 clusters")
    chk("Singapore: cluster weeks fall in Feb-Jul 2020",
        lambda: pd.to_datetime(cz.week_end).min() >= pd.Timestamp("2020-02-01") and
                pd.to_datetime(cz.week_end).max() <= pd.Timestamp("2020-07-31"), "date format problem in A11")
    for f in ["global_country_year.csv", "global_country_month.csv", "disease_info.csv", "gdo_current_season.csv",
              "sg_weekly_disease.csv", "sg_population.csv", "sg_monthly_lagged.csv", "sg_clusters_2020.csv",
              "sg_subzone_profile.csv", "mp2019_subzones.geojson", "nea_clusters_now.geojson"]:
        chk(f"file exists: {f}", lambda f=f: os.path.getsize(f"{OUT}/{f}") > 0, "see the FAIL message for that step above")
    print("\nCHECK REPORT")
    for status, name, fix in rows:
        print(f"{status}  {name}" + (f"\n      fix: {fix}" if fix else ""))
    print(f"\n{sum(r[0] == 'PASS' for r in rows)} of {len(rows)} checks passed.")
    return rows


def zip_outputs(name="cleaned_data.zip"):
    zpath = f"{BASE}/{name}"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(os.listdir(OUT)):
            z.write(f"{OUT}/{f}", f)
    print("Zipped:", zpath)
    if IN_COLAB:
        from google.colab import files
        files.download(zpath)
    return zpath


REPORT = check_report()
zip_outputs()

# Part B (optional, about 45–60 min): HDB building age per subzone
Looks up each HDB block's location with OneMap, then takes the median completion year per subzone.
Progress is saved to your Drive every 200 blocks, so if Colab disconnects, just run all again and it continues where it stopped.

In [ ]:
def onemap_lookup(blk, street):
    r = http_get("https://www.onemap.gov.sg/api/common/elastic/search",
                 {"searchVal": f"{blk} {street}", "returnGeom": "Y", "getAddrDetails": "Y", "pageNum": 1}, tries=3, timeout=30)
    res = r.json().get("results", [])
    best = next((x for x in res if str(x.get("BLK_NO", "")).upper() == str(blk).upper()), res[0] if res else None)
    return (float(best["LATITUDE"]), float(best["LONGITUDE"])) if best else (np.nan, np.nan)


if RUN_PART_B_HDB_GEOCODING and subz is not None and profile is not None:
    from concurrent.futures import ThreadPoolExecutor
    hdb = datagovsg_table("d_17f5382f26140b1fdae0ba2ef6239d2f", "hdb_property_information")
    hdb = hdb[hdb["residential"] == "Y"].copy()
    hdb["year_completed"] = num(hdb["year_completed"])
    hdb = hdb[hdb["year_completed"] <= 2020]
    cache_path = f"{RAW}/hdb_geocoded.csv"
    done = pd.read_csv(cache_path, dtype={"blk_no": str}) if os.path.exists(cache_path) else pd.DataFrame(
        columns=["blk_no", "street", "lat", "lon"])
    todo = hdb.merge(done[["blk_no", "street"]], on=["blk_no", "street"], how="left", indicator=True)
    todo = todo[todo["_merge"] == "left_only"][["blk_no", "street"]].drop_duplicates()
    print(f"{len(done)} blocks already located, {len(todo)} to go (about {len(todo) / 240:.0f} minutes).")

    def work(row):
        time.sleep(0 if TEST_MODE else 1.0)          # stays under OneMap's ~250 requests per minute
        try:
            la, lo = onemap_lookup(row[0], row[1])
        except Exception:
            la, lo = np.nan, np.nan
        return {"blk_no": row[0], "street": row[1], "lat": la, "lon": lo}

    batch = []
    with ThreadPoolExecutor(max_workers=4) as ex:
        for i, res in enumerate(ex.map(work, todo.itertuples(index=False, name=None)), 1):
            batch.append(res)
            if i % 200 == 0 or i == len(todo):
                done = pd.concat([done, pd.DataFrame(batch)], ignore_index=True)
                done.to_csv(cache_path, index=False)
                batch = []
                print(f"  {i}/{len(todo)} located")

    geo = hdb.merge(done, on=["blk_no", "street"], how="inner").dropna(subset=["lat", "lon"])
    pts = gpd.GeoDataFrame(geo, geometry=gpd.points_from_xy(geo["lon"], geo["lat"]), crs="EPSG:4326")
    geo["subzone"] = to_subzone(pts).values
    med = geo.groupby("subzone")["year_completed"].median()
    profile = pd.read_csv(f"{OUT}/sg_subzone_profile.csv")
    profile["hdb_median_year_completed"] = profile["subzone"].map(med)
    profile["hdb_median_age_2020"] = 2020 - profile["hdb_median_year_completed"]
    profile.to_csv(f"{OUT}/sg_subzone_profile.csv", index=False)
    note("B sg_subzone_profile.csv building age", True,
         f"{len(geo)} of {len(hdb)} blocks located; {profile.hdb_median_year_completed.notna().sum()} subzones have an age")
    zip_outputs("cleaned_data_final.zip")
else:
    print("Part B skipped (switch RUN_PART_B_HDB_GEOCODING to True to run it).")

# Part C (optional): country humidity from ERA5
Two ways to get the ERA5 file. Either way, set `RUN_PART_C_ERA5_HUMIDITY = True` in Settings and Run all.

**1. Let the notebook download it.** Create a free account at https://cds.climate.copernicus.eu, open the dataset
"ERA5 monthly averaged data on single levels from 1940 to present", accept its licence at the bottom of the
Download tab, then copy your personal access token from your profile page into `CDS_API_KEY` in Settings.
The request can wait in Copernicus's queue for a while before it starts.

**2. Use a file already downloaded from that page.** It must be NetCDF, "Monthly averaged reanalysis", with both
"2m temperature" and "2m dewpoint temperature", every month of 2000–2024, whole world. Rename it exactly
`era5_t2m_d2m_monthly_1deg.nc` (even if it is really a .zip) and put it in `G1T8_mosquito_data/raw`. No token needed.
A 0.25° file works too: the notebook averages it to 1° first.

In [ ]:
def relative_humidity(t_c, td_c):
    """Relative humidity (%) from temperature and dew point in deg C (Magnus formula)."""
    return 100 * np.exp(17.625 * td_c / (243.04 + td_c)) / np.exp(17.625 * t_c / (243.04 + t_c))


def era5_country_humidity(nc_path, countries_path):
    import xarray as xr
    ds = xr.open_dataset(nc_path)
    if "expver" in ds.dims:                 # recent months can come split into ERA5 / ERA5T versions
        ds = ds.mean("expver", skipna=True)
    if "number" in ds.dims:
        ds = ds.isel(number=0)
    tname = "valid_time" if "valid_time" in ds.dims else "time"
    lat = "latitude" if "latitude" in ds.dims else "lat"
    lon = "longitude" if "longitude" in ds.dims else "lon"
    step = abs(float(ds[lat].values[1] - ds[lat].values[0]))
    if step < 0.9:                          # a 0.25° file from the website: average to about 1° so Colab's memory copes
        k = int(round(1.0 / step))
        ds = ds.coarsen({lat: k, lon: k}, boundary="trim").mean()
    t, td = ds["t2m"] - 273.15, ds["d2m"] - 273.15
    rhv = relative_humidity(t, td).transpose(tname, lat, lon)
    lons = ((ds[lon].values + 180) % 360) - 180
    LON, LAT = np.meshgrid(lons, ds[lat].values)
    cells = gpd.GeoDataFrame({"cell": np.arange(LON.size), "w": np.cos(np.deg2rad(LAT.ravel()))},
                             geometry=gpd.points_from_xy(LON.ravel(), LAT.ravel()), crs="EPSG:4326")
    world = gpd.read_file(countries_path)
    world["iso3"] = np.where(world["ISO_A3"].astype(str).str.fullmatch(r"[A-Z]{3}"), world["ISO_A3"], world["ADM0_A3"])
    hit = gpd.sjoin(cells, world[["iso3", "geometry"]], how="inner", predicate="within")
    flat = rhv.values.reshape(rhv.shape[0], -1)          # time x cells
    years = pd.to_datetime(ds[tname].values).year
    out = []
    for iso3, grp in hit.groupby("iso3"):
        idx, w = grp["cell"].values, grp["w"].values
        series = np.nansum(flat[:, idx] * w, axis=1) / w.sum()
        out.append(pd.DataFrame({"iso3": iso3, "year": years, "rh": series}))
    df = pd.concat(out).groupby(["iso3", "year"], as_index=False)["rh"].mean().rename(columns={"rh": "rh_pct"})
    return df


if RUN_PART_C_ERA5_HUMIDITY:
    try:
        nc = f"{RAW}/era5_t2m_d2m_monthly_1deg.nc"
        if not os.path.exists(nc):
            import cdsapi
            client = cdsapi.Client(url="https://cds.climate.copernicus.eu/api", key=CDS_API_KEY)
            client.retrieve("reanalysis-era5-single-levels-monthly-means", {
                "product_type": ["monthly_averaged_reanalysis"],
                "variable": ["2m_temperature", "2m_dewpoint_temperature"],
                "year": [str(y) for y in range(Y0, Y1 + 1)],
                "month": [f"{m:02d}" for m in range(1, 13)],
                "time": ["00:00"], "grid": [1.0, 1.0],
                "data_format": "netcdf", "download_format": "unarchived"}, nc)
        if zipfile.is_zipfile(nc):
            with zipfile.ZipFile(nc) as z:
                z.extractall(f"{RAW}/era5_parts")
            import xarray as xr
            parts = [xr.open_dataset(f"{RAW}/era5_parts/{n}") for n in os.listdir(f"{RAW}/era5_parts") if n.endswith(".nc")]
            xr.merge(parts).to_netcdf(f"{RAW}/era5_merged.nc")
            nc = f"{RAW}/era5_merged.nc"
        borders = raw_file("ne_50m_admin_0_countries.zip",
                           "https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_countries.zip")
        hum = era5_country_humidity(nc, borders)
        hum.to_csv(f"{RAW}/era5_rh_country_year.csv", index=False)
        gy = pd.read_csv(f"{OUT}/global_country_year.csv").drop(columns="rh_pct")
        gy = gy.merge(hum, on=["iso3", "year"], how="left")
        gy = fill_singapore_climate(gy)        # Singapore is smaller than one grid cell: use Changi humidity
        cols = list(pd.read_csv(f"{OUT}/global_country_year.csv", nrows=1).columns)
        gy[cols].to_csv(f"{OUT}/global_country_year.csv", index=False)
        note("C humidity", True, f"{hum.iso3.nunique()} countries")
        zip_outputs("cleaned_data_final.zip")
    except Exception as e:
        note("C humidity", False, str(e))
else:
    print("Part C skipped (switch RUN_PART_C_ERA5_HUMIDITY to True and add your CDS_API_KEY to run it).")